


#**Instacart Market Basket Analysis using Apache Spark**
(Big Data & NoSQL Databases – Assignment 2)

#Environment Setup and Spark Session Initialization

In [1]:
# Installing required packages
!pip install pyspark
!pip install findspark
!pip install pandas

In [2]:
import findspark
findspark.init()

In [3]:
import pandas as pd
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [4]:
# Creating a spark context class
sc = SparkContext()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Python Spark DataFrames basic example") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

In [5]:
spark

In [6]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'instacart-market-basket-analysis' dataset.
Path to dataset files: /kaggle/input/instacart-market-basket-analysis


#**Data Loading**



In [7]:
# Products
products_df = spark.read.csv(f"{path}/products.csv", header=True, inferSchema=True)

# Aisles
aisles_df = spark.read.csv(f"{path}/aisles.csv", header=True, inferSchema=True)

# Departments
departments_df = spark.read.csv(f"{path}/departments.csv", header=True, inferSchema=True)

# Orders
orders_df = spark.read.csv(f"{path}/orders.csv", header=True, inferSchema=True)

# Order-Product (prior orders)
order_products_prior_df = spark.read.csv(
    f"{path}/order_products__prior.csv", header=True, inferSchema=True
)

# Order-Product (train set)
order_products_train_df = spark.read.csv(
    f"{path}/order_products__train.csv", header=True, inferSchema=True
)

# **Data Preprocessing**

**Handling Nulls**

In [8]:
from pyspark.sql.functions import col, count, when


def check_missing(df, name):
    print(f"\nMissing values in {name}:")

    df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ]).show()

# Run checks
check_missing(products_df, "products")
check_missing(aisles_df, "aisles")
check_missing(departments_df, "departments")
check_missing(orders_df, "orders")
check_missing(order_products_prior_df, "order_products_prior")
check_missing(order_products_train_df, "order_products_train")


Missing values in products:
+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
|         0|           0|       0|            0|
+----------+------------+--------+-------------+


Missing values in aisles:
+--------+-----+
|aisle_id|aisle|
+--------+-----+
|       0|    0|
+--------+-----+


Missing values in departments:
+-------------+----------+
|department_id|department|
+-------------+----------+
|            0|         0|
+-------------+----------+


Missing values in orders:
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------

In [9]:
orders_df = orders_df.na.fill({
    "days_since_prior_order": 0
})

In [10]:
from pyspark.sql.functions import col, count, when


def check_missing(df, name):
    print(f"\nMissing values in {name}:")

    df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ]).show()

# Run checks
check_missing(products_df, "products")
check_missing(aisles_df, "aisles")
check_missing(departments_df, "departments")
check_missing(orders_df, "orders")
check_missing(order_products_prior_df, "order_products_prior")
check_missing(order_products_train_df, "order_products_train")


Missing values in products:
+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
|         0|           0|       0|            0|
+----------+------------+--------+-------------+


Missing values in aisles:
+--------+-----+
|aisle_id|aisle|
+--------+-----+
|       0|    0|
+--------+-----+


Missing values in departments:
+-------------+----------+
|department_id|department|
+-------------+----------+
|            0|         0|
+-------------+----------+


Missing values in orders:
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                     0|
+--------+-------+--------+------------+---------

**Handling Duplicates**

In [11]:

def check_duplicates(df, name, key_cols=None):
    print(f"\n===== {name} =====")

    total_count = df.count()

    # If key columns are provided → check logical duplicates
    if key_cols:
        distinct_count = df.dropDuplicates(key_cols).count()
    else:
        # fallback → full row duplicates
        distinct_count = df.dropDuplicates().count()

    print(f"Total rows     : {total_count}")
    print(f"Distinct rows  : {distinct_count}")
    print(f"Duplicate rows : {total_count - distinct_count}")


# =========================================================
# RUN DUPLICATE CHECKS (USE BUSINESS KEYS)
# =========================================================

check_duplicates(products_df, "products", ["product_id"])
check_duplicates(aisles_df, "aisles", ["aisle_id"])
check_duplicates(departments_df, "departments", ["department_id"])

check_duplicates(orders_df, "orders", ["order_id"])

check_duplicates(order_products_prior_df, "order_products_prior", ["order_id", "product_id"])
check_duplicates(order_products_train_df, "order_products_train", ["order_id", "product_id"])


===== products =====
Total rows     : 49688
Distinct rows  : 49688
Duplicate rows : 0

===== aisles =====
Total rows     : 134
Distinct rows  : 134
Duplicate rows : 0

===== departments =====
Total rows     : 21
Distinct rows  : 21
Duplicate rows : 0

===== orders =====
Total rows     : 3421083
Distinct rows  : 3421083
Duplicate rows : 0

===== order_products_prior =====
Total rows     : 32434489
Distinct rows  : 32434489
Duplicate rows : 0

===== order_products_train =====
Total rows     : 1384617
Distinct rows  : 1384617
Duplicate rows : 0


**Data Type Cleaning**

In [12]:
from pyspark.sql.functions import col, expr

def to_int(df, col_name):
    # Use try_cast to gracefully handle malformed string values by converting them to NULL
    return df.withColumn(col_name, expr(f"try_cast({col_name} as int)"))


orders_df = to_int(orders_df, "order_id")
orders_df = to_int(orders_df, "user_id")

order_products_prior_df = to_int(order_products_prior_df, "order_id")
order_products_prior_df = to_int(order_products_prior_df, "product_id")
order_products_prior_df = to_int(order_products_prior_df, "add_to_cart_order") # Apply to_int
order_products_prior_df = to_int(order_products_prior_df, "reordered") # Apply to_int

order_products_train_df = to_int(order_products_train_df, "order_id")
order_products_train_df = to_int(order_products_train_df, "product_id")
order_products_train_df = to_int(order_products_train_df, "add_to_cart_order") # Apply to_int
order_products_train_df = to_int(order_products_train_df, "reordered") # Apply to_int

products_df = to_int(products_df, "product_id")
products_df = to_int(products_df, "aisle_id")
products_df = to_int(products_df, "department_id")

aisles_df = to_int(aisles_df, "aisle_id")
departments_df = to_int(departments_df, "department_id")

**Data Integration (Joining Orders, Products, Aisles, and Departments)**

In [13]:
from pyspark.sql.functions import col

# 1. Union prior + train (same schema)
order_products_df = order_products_prior_df.unionByName(order_products_train_df)


# 2. Join all tables step by step
merged_df = order_products_df \
    .join(orders_df, on="order_id", how="inner") \
    .join(products_df, on="product_id", how="inner") \
    .join(aisles_df, on="aisle_id", how="inner") \
    .join(departments_df, on="department_id", how="inner")


# 3. Inspect results
print("Schema of the final merged_df:")
merged_df.printSchema()

print("First 5 rows of the final merged_df:")
merged_df.show(5, truncate=False)

Schema of the final merged_df:
root
 |-- department_id: integer (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = false)
 |-- product_name: string (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)

First 5 rows of the final merged_df:
+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+----------------------------------+----------------+----------+
|department_id|aisle_id|product_id|order_id|add_to_

**Derived Columns**

In [14]:
from pyspark.sql.functions import count, sum as _sum, col

# ================================
# 1. Total Items Per Order
# ================================
items_per_order = merged_df.groupBy("order_id") \
    .agg(count("*").alias("total_items"))

merged_df = merged_df.join(items_per_order, on="order_id", how="left")


# ================================
# 2. Reorder Ratio
# ================================
reorder_ratio = merged_df.groupBy("order_id") \
    .agg(
        (_sum("reordered") / count("*")).alias("reorder_ratio")
    )

merged_df = merged_df.join(reorder_ratio, on="order_id", how="left")


# ================================
# 3. Customer Order Frequency
# ================================
order_frequency = merged_df.groupBy("user_id") \
    .agg(count("order_id").alias("order_frequency"))

merged_df = merged_df.join(order_frequency, on="user_id", how="left")


# ================================
# Preview Results
# ================================
merged_df.select(
    "order_id",
    "user_id",
    "total_items",
    "reorder_ratio",
    "order_frequency"
).show(10, truncate=False)

+--------+-------+-----------+------------------+---------------+
|order_id|user_id|total_items|reorder_ratio     |order_frequency|
+--------+-------+-----------+------------------+---------------+
|14      |18194  |11         |0.8181818181818182|403            |
|14      |18194  |11         |0.8181818181818182|403            |
|14      |18194  |11         |0.8181818181818182|403            |
|14      |18194  |11         |0.8181818181818182|403            |
|14      |18194  |11         |0.8181818181818182|403            |
|14      |18194  |11         |0.8181818181818182|403            |
|1       |112108 |8          |0.5               |29             |
|1       |112108 |8          |0.5               |29             |
|1       |112108 |8          |0.5               |29             |
|1       |112108 |8          |0.5               |29             |
+--------+-------+-----------+------------------+---------------+
only showing top 10 rows


# **DataFrame Analysis**

**Query A**

In [15]:
from pyspark.sql.functions import count

merged_df.groupBy("department") \
    .agg(count("*").alias("total_orders")) \
    .orderBy("total_orders", ascending=False) \
    .limit(10) \
    .show()


+---------------+------------+
|     department|total_orders|
+---------------+------------+
|        produce|     9888378|
|     dairy eggs|     5631067|
|         snacks|     3006412|
|      beverages|     2804175|
|         frozen|     2336858|
|         pantry|     1956819|
|         bakery|     1225181|
|   canned goods|     1114857|
|           deli|     1095540|
|dry goods pasta|      905340|
+---------------+------------+



The results show that produce is by far the most ordered department (9.8M orders), followed by dairy & eggs and snacks. This indicates that fresh food and essential grocery categories dominate customer demand on Instacart.

From a business perspective, this suggests that the company should prioritize strong inventory management and supply chain efficiency for produce and dairy products, as they are core drivers of platform usage. Additionally, these high-demand categories can be strategically used in promotions and app recommendations to maximize sales and customer engagement.

**Query B**

In [16]:
from pyspark.sql.functions import sum as _sum, count

merged_df.groupBy("department") \
    .agg((_sum("reordered") / count("*")).alias("reorder_rate")) \
    .orderBy("reorder_rate", ascending=False) \
    .show()

+---------------+-------------------+
|     department|       reorder_rate|
+---------------+-------------------+
|     dairy eggs| 0.6701612678378716|
|      beverages| 0.6536510738452486|
|        produce|  0.650520843762243|
|         bakery| 0.6283806229446914|
|           deli| 0.6081302371433266|
|           pets| 0.6025572044883145|
|         babies| 0.5776798718156188|
|           bulk| 0.5770900590003339|
|         snacks| 0.5744638459399444|
|        alcohol| 0.5712205105025927|
|   meat seafood| 0.5686247189673691|
|      breakfast| 0.5613508346311373|
|         frozen| 0.5426337415452714|
|dry goods pasta| 0.4622197185587735|
|   canned goods| 0.4586390900357624|
|          other|0.40705246022160374|
|      household| 0.4033400933842295|
|        missing| 0.3943227040157114|
|  international| 0.3696822037666056|
|         pantry| 0.3474000405760574|
+---------------+-------------------+
only showing top 20 rows


The results show that dairy & eggs, beverages, and produce have the highest reorder rates (~0.65–0.67), indicating that customers frequently repurchase these items. In contrast, departments such as pantry, international, and missing have significantly lower reorder rates.

From a business perspective, high reorder rates indicate strong customer loyalty and habitual purchasing behavior, especially for essential goods. These departments are ideal for subscription services, automated reordering features, and personalized recommendations. Low-reorder departments may require targeted promotions to increase customer engagement.

**Query C**

In [17]:
from pyspark.sql.functions import countDistinct

merged_df.groupBy("order_hour_of_day") \
    .agg(countDistinct("order_id").alias("total_orders")) \
    .orderBy("total_orders", ascending=False) \
    .show()

+-----------------+------------+
|order_hour_of_day|total_orders|
+-----------------+------------+
|               10|      282470|
|               11|      278616|
|               15|      277207|
|               14|      276659|
|               13|      271885|
|               12|      266828|
|               16|      266444|
|                9|      252529|
|               17|      223433|
|               18|      178556|
|                8|      174664|
|               19|      137341|
|               20|      102087|
|                7|       90032|
|               21|       76486|
|               22|       59982|
|               23|       39139|
|                6|       29913|
|                0|       22224|
|                1|       12103|
+-----------------+------------+
only showing top 20 rows


The results show that most orders are placed between 10 AM and 4 PM, with peak activity around 10–11 AM and early afternoon hours. Order volume gradually decreases in the evening and reaches its lowest levels during late night and early morning hours.

This helps Instacart optimize staffing, delivery scheduling, and warehouse operations by focusing resources during peak shopping hours.

**Query D**

In [18]:
from pyspark.sql.functions import avg

merged_df.agg(avg("total_items").alias("avg_basket_size")).show()

+------------------+
|   avg_basket_size|
+------------------+
|15.735475331796943|
+------------------+



The average basket size is about 15.7 items per order, meaning customers usually buy around 16 products per purchase.

This helps Instacart plan delivery capacity, optimize warehouse operations, and design promotions to increase order size

**Query E**

In [19]:
merged_df.filter("reordered = 1") \
    .groupBy("product_name") \
    .agg(count("*").alias("reorder_count")) \
    .orderBy("reorder_count", ascending=False) \
    .limit(10) \
    .show()

+--------------------+-------------+
|        product_name|reorder_count|
+--------------------+-------------+
|              Banana|       415166|
|Bag of Organic Ba...|       329275|
|Organic Strawberries|       214448|
|Organic Baby Spinach|       194939|
|Organic Hass Avocado|       176173|
|     Organic Avocado|       140270|
|  Organic Whole Milk|       118684|
|         Large Lemon|       112178|
| Organic Raspberries|       109688|
|        Strawberries|       104588|
+--------------------+-------------+



The most frequently reordered products are dominated by fresh and organic items such as bananas, organic strawberries, organic spinach, and avocados. These products show extremely high reorder counts, with bananas leading significantly at over 415,000 reorders.

From a business perspective, these items represent everyday essentials and healthy lifestyle preferences, making them highly predictable demand drivers. Instacart should ensure continuous availability of these products and may consider highlighting them in personalized recommendations or subscription-based grocery bundles.

**Query F**

In [20]:
from pyspark.sql.functions import round

merged_df.groupBy("department") \
    .agg(round((count("*") * 100.0 / merged_df.count()), 4).alias("percentage_contribution")) \
    .orderBy("percentage_contribution", ascending=False) \
    .show()

+---------------+-----------------------+
|     department|percentage_contribution|
+---------------+-----------------------+
|        produce|                 29.239|
|     dairy eggs|                16.6506|
|         snacks|                 8.8897|
|      beverages|                 8.2917|
|         frozen|                 6.9099|
|         pantry|                 5.7861|
|         bakery|                 3.6227|
|   canned goods|                 3.2965|
|           deli|                 3.2394|
|dry goods pasta|                  2.677|
|      household|                 2.2906|
|   meat seafood|                 2.1859|
|      breakfast|                 2.1854|
|  personal care|                 1.3859|
|         babies|                 1.2973|
|  international|                 0.8313|
|        alcohol|                  0.471|
|           pets|                 0.3023|
|        missing|                 0.2289|
|          other|                 0.1126|
+---------------+-----------------

The results show that produce alone contributes nearly 29% of total sales, making it the dominant department by a large margin. Dairy & eggs also contribute significantly at around 16.6%, followed by snacks and beverages.

From a business perspective, this indicates that a small number of core departments drive the majority of revenue. Instacart should focus on strengthening supply chains, marketing efforts, and promotions around high-contribution departments, while also exploring strategies to grow underperforming categories to diversify revenue sources.

#User Segmentation (Bonus Task)

In [21]:
from pyspark.sql.functions import countDistinct, when, col

# Step 1: Compute number of orders per user (safe definition)
user_orders = merged_df.groupBy("user_id") \
    .agg(countDistinct("order_id").alias("order_frequency"))

# Step 2: Create user segments
user_segmented = user_orders.withColumn(
    "user_segment",
    when(col("order_frequency") <= 10, "Low Activity")
    .when(col("order_frequency") <= 30, "Medium Activity")
    .otherwise("High Activity")
)

# Step 3: Count users per segment
user_segmented.groupBy("user_segment") \
    .agg(countDistinct("user_id").alias("num_users")) \
    .orderBy("num_users", ascending=False) \
    .show()

+---------------+---------+
|   user_segment|num_users|
+---------------+---------+
|   Low Activity|   107431|
|Medium Activity|    70139|
|  High Activity|    28639|
+---------------+---------+

